# Búsqueda local

Con la solución del greedy, buscamos ahora hacer una búsqueda local para intentar mejorar los resultados

In [1]:
import pandas as pd 
from importlib import reload

import Clases.asignacion as asignacion_module
reload(asignacion_module)
from Clases.asignacion import Asignacion

import Clases.caja as caja_module
reload(caja_module)
from Clases.caja import Caja

import Clases.producto as producto_module
reload(producto_module)
from Clases.producto import Producto

import Clases.solucion as solucion_module
reload(solucion_module)
from Clases.solucion import Solucion

catalogo_productos = pd.read_csv("Datos-finales/catalogo_productos.csv")
operaciones_planta = pd.read_csv("Datos-finales/operaciones_planta.csv")

cajas_nuevas = pd.read_csv("4r.cajas_nuevas.csv")
factibilidad = pd.read_csv("Factibilidad/factibilidad_3mm.csv")

solucion = pd.read_csv('Soluciones/solucion7-greedy_3mm_mejor_sol_mas_cajas.csv')

Empecemos guardando los productos y tipos de cajas en listas en el estado actual, para cargarlos luego a las soluciones. Almacenamos también las cajas asignables a cada producto en un diccionario.

In [2]:
def guardar_cajas_y_productos(grosor=3):
    
    cajas = {
        row["caja_tipo_id"]: Caja(
            caja_id=row["caja_tipo_id"],
            dim_interior_ancho=row["caja_interior_ancho"],
            dim_interior_largo=row["caja_interior_largo"],
            dim_interior_alto=row["caja_interior_alto"]
        )
        for _, row in cajas_nuevas.iterrows()
    }

    prod_op_merge = catalogo_productos.merge(operaciones_planta, on="codigo_producto")
    productos = {
        row["codigo_producto"]: Producto(
            codigo_producto = row['codigo_producto'],
            cantidad_paquetes = row['cantidad_paquetes'],
            peso_paquete = row['peso_neto_paquete'],
            demanda_buenos_aires = row['volumen_producto_planta_buenos_aires'],
            demanda_curitiba = row['volumen_producto_planta_curitiba'],
            demanda_santiago = row['volumen_producto_planta_santiago'],
            demanda_monterrey = row['volumen_producto_planta_monterrey'],
            demanda_bakersfield = row['volumen_producto_planta_bakersfield'],
            dim_producto_ancho = row['dim_producto_ancho'], 
            dim_producto_largo = row['dim_producto_largo'],
            dim_producto_alto = row['dim_producto_alto']
        )
        for _, row in prod_op_merge.iterrows()
    }
    
    cajas_asignables_por_producto = {}

    for codigo, group in factibilidad.groupby('codigo_producto'):
        # Obtener IDs de los tipos de cajas
        cajas_ids_unicos = list(group['caja_tipo_id'].unique())
        
        cajas_producto = []
        for caja_id in cajas_ids_unicos:
            cajas_producto.append(caja_id)
            
        cajas_asignables_por_producto[codigo] = cajas_producto
                
    # Elegir grosor
    for caja_id, caja in cajas.items():
        caja.elegir_grosor(grosor_mm=grosor)
        
    return cajas, productos, cajas_asignables_por_producto

#### **Reconstrucción de la solución inicial (Greedy)**

El csv exportado por `exportar_submmit` solo guarda las dimensiones *exteriores* de la caja asignada a cada producto, no el `caja_tipo_id`. Para poder operar con objetos `Caja` (y sus descuentos por volumen), reconstruimos el `caja_tipo_id` real restando el grosor a las dimensiones exteriores y buscando la caja correspondiente entre las cajas ya creadas.

In [3]:
grosor = 3
cajas, productos, cajas_asignables_por_producto = guardar_cajas_y_productos(grosor=grosor)

# Índice de cajas por sus dimensiones interiores (redondeadas), para poder
# encontrar el caja_tipo_id real a partir de las dimensiones exteriores del csv
indice_cajas_por_dim = {
    (c.dim_interior_ancho, c.dim_interior_largo, c.dim_interior_alto): c
    for c in cajas.values()
}

solucion_inicial = Solucion(grosor, "Greedy 5 (maximizar utilización de pallet) - punto de partida")

for _, row in solucion.iterrows():
    producto = productos[row["codigo_producto"]]

    dim_ancho = row["caja_exterior_ancho"] - 2 * row["caja_grosor_mm"]
    dim_largo = row["caja_exterior_largo"] - 2 * row["caja_grosor_mm"]
    dim_alto = row["caja_exterior_alto"] - 2 * row["caja_grosor_mm"]

    caja = indice_cajas_por_dim[(dim_ancho, dim_largo, dim_alto)]

    asignacion = Asignacion(producto, caja)
    solucion_inicial.agregar_asignacion(asignacion)

solucion_inicial.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30166293.939999998
Costo flete: 179068800
Costo total: 209235093.94
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 3mm
Criterio elegido: Greedy 5 (maximizar utilización de pallet) - punto de partida
Número de tipos de cajas distintos: 76
Costo packaging: 27406334.33999998
Costo flete: 161380650
Costo total: 188786984.33999997
Utilización de pallet promedio: 0.9593515126880586
Utilización de caja promedio: 0.9445527162171283
Ahorro costo total: 9.77279%


#### **Cálculo del delta de costo de un movimiento**

El costo de packaging de una caja depende del volumen acumulado de **todos** los productos que la usan (por los descuentos por volumen). Por eso, mover un producto de una caja a otra no tiene un delta de costo aislado: afecta también el costo de los demás productos que ya comparten esa caja.

La función `calcular_delta_costo` simula el movimiento (usando los métodos ya existentes `asignar_producto` / `revocar_producto` de `Caja`, que actualizan los descuentos), mide el cambio real en el costo total de las dos cajas involucradas, y **revierte la simulación** antes de devolver el resultado. El costo de flete sí es independiente del resto de los productos de la caja (depende solo de la demanda del producto y de `cantidad_cajas_por_pallet` de la caja), así que se calcula aparte, sin necesidad de simular nada.

Nota: no volvemos a chequear factibilidad (dimensión, headspace, resistencia, utilización de pallet ≥ 0.6) porque `cajas_asignables_por_producto` ya viene filtrado por esos criterios desde `5_factibilidad.ipynb`.

In [4]:
def calcular_delta_costo(producto, caja_actual, caja_nueva):
    """
    Calcula cuánto cambiaría el costo total (packaging + flete) si se
    reasignara `producto` de `caja_actual` a `caja_nueva`.
    Simula el movimiento y lo revierte: no deja efectos secundarios.
    """
    # --- Costo de packaging (afecta a todos los productos de ambas cajas) ---
    costo_antes = caja_actual.costo_packaging_total() + caja_nueva.costo_packaging_total()

    caja_actual.revocar_producto(producto)
    caja_nueva.asignar_producto(producto)

    costo_despues = caja_actual.costo_packaging_total() + caja_nueva.costo_packaging_total()

    # Revertimos la simulación para dejar el estado como estaba
    caja_nueva.revocar_producto(producto)
    caja_actual.asignar_producto(producto)

    delta_packaging = costo_despues - costo_antes

    # --- Costo de flete (independiente de otros productos de la caja) ---
    pallets_actual = Asignacion(producto, caja_actual).cant_pallets_requeridas()
    pallets_nueva = Asignacion(producto, caja_nueva).cant_pallets_requeridas()
    delta_flete = 150 * (pallets_nueva - pallets_actual)

    return delta_packaging + delta_flete

#### **Búsqueda local: mejor mejora por pasada**

En cada pasada se evalúa reasignar **cada producto** a cada una de sus cajas alternativas factibles, y se aplica el movimiento con la mayor reducción de costo total encontrado en esa pasada. Se repite hasta que no se encuentra ninguna mejora (óptimo local) o se alcanza `max_iteraciones`.

Por ahora el único tipo de movimiento es "reasignar un producto a otra caja"; más adelante se puede extender con otros vecindarios (por ejemplo, intercambio de cajas entre pares de productos).

⚠️ **Nota de performance:** según `5_factibilidad.ipynb`, algunos productos tienen miles de cajas candidatas asignables. Evaluar *todas* las combinaciones producto×caja en cada pasada puede ser lento en instancias grandes. Si se vuelve impráctico, se puede limitar la cantidad de candidatos evaluados por producto por pasada (parámetro `max_candidatos_por_producto`, que si se deja en `None` evalúa todos).

In [ ]:
import random

def busqueda_local(solucion, cajas, cajas_asignables_por_producto, max_iteraciones=50,
                    max_candidatos_por_producto=None, verbose=True, semilla=42):
    """
    Búsqueda local por 'mejor mejora' (steepest descent) sobre reasignaciones
    individuales de caja. Modifica `solucion` in-place y devuelve un DataFrame
    con el historial de movimientos aplicados.

    Si se interrumpe manualmente (KeyboardInterrupt), los movimientos ya
    aplicados quedan y se devuelve igual el historial parcial armado hasta
    ese momento, en vez de perderse.
    """
    random.seed(semilla)
    historial = []
    costo_actual = solucion.costo_total()
    UMBRAL_MEJORA = 1e-6  # tolerancia para evitar ciclos por ruido de punto flotante
    iteracion = 0

    try:
        for iteracion in range(1, max_iteraciones + 1):
            mejor_delta = -UMBRAL_MEJORA
            mejor_movimiento = None  # (asignacion, caja_nueva)

            for asignacion in solucion.asignaciones:
                producto = asignacion.producto
                caja_actual = asignacion.caja
                candidatos = cajas_asignables_por_producto.get(producto.codigo_producto, [])

                if max_candidatos_por_producto is not None and len(candidatos) > max_candidatos_por_producto:
                    candidatos = random.sample(candidatos, max_candidatos_por_producto)

                for caja_id_candidata in candidatos:
                    if caja_id_candidata == caja_actual.caja_id:
                        continue

                    caja_candidata = cajas[caja_id_candidata]
                    delta = calcular_delta_costo(producto, caja_actual, caja_candidata)

                    if delta < mejor_delta:
                        mejor_delta = delta
                        mejor_movimiento = (asignacion, caja_candidata)

            if mejor_movimiento is None:
                if verbose:
                    print(f"Iteración {iteracion}: no se encontraron mejoras. Óptimo local alcanzado.")
                break

            # Aplicamos el mejor movimiento encontrado en esta pasada
            asignacion, caja_nueva = mejor_movimiento
            producto = asignacion.producto
            caja_vieja = asignacion.caja

            caja_vieja.revocar_producto(producto)
            caja_nueva.asignar_producto(producto)
            asignacion.caja = caja_nueva

            # Actualizamos el tracking de tipos de caja utilizados en la solución
            if caja_nueva not in solucion.tipos_cajas_utilizados:
                solucion.tipos_cajas_utilizados.append(caja_nueva)
                solucion.cantidad_tipos_cajas += 1
            if len(caja_vieja.productos_asignados) == 0 and caja_vieja in solucion.tipos_cajas_utilizados:
                solucion.tipos_cajas_utilizados.remove(caja_vieja)
                solucion.cantidad_tipos_cajas -= 1

            costo_actual += mejor_delta

            historial.append({
                "iteracion": iteracion,
                "codigo_producto": producto.codigo_producto,
                "caja_anterior": caja_vieja.caja_id,
                "caja_nueva": caja_nueva.caja_id,
                "delta_costo": mejor_delta,
                "costo_total_estimado": costo_actual
            })

            if verbose:
                print(f"Iteración {iteracion}: {producto.codigo_producto} "
                      f"{caja_vieja.caja_id} -> {caja_nueva.caja_id} "
                      f"(Δ={mejor_delta:,.2f} | costo total ≈ {costo_actual:,.2f})")

    except KeyboardInterrupt:
        if verbose:
            print(f"\nInterrumpido manualmente en la iteración {iteracion}. "
                  f"Se conservan los {len(historial)} movimientos ya aplicados.")

    return pd.DataFrame(historial)

#### **Ejecutamos la búsqueda local**

In [8]:
historial_busqueda_local = busqueda_local(
    solucion_inicial,
    cajas,
    cajas_asignables_por_producto,
    max_iteraciones=50,          # subir si todavía encuentra mejoras al llegar al límite
    max_candidatos_por_producto=None,  # poner un número (ej. 200) si es muy lento
    verbose=True
)

historial_busqueda_local

Iteración 1: BR0239 CAJ2229012 -> CAJ2132580 (Δ=-50,991.00 | costo total ≈ 188,735,993.34)
Iteración 2: BR0321 CAJ1942558 -> CAJ1955788 (Δ=-49,950.00 | costo total ≈ 188,686,043.34)
Iteración 3: BR0159 CAJ0708749 -> CAJ0443990 (Δ=-36,403.14 | costo total ≈ 188,649,640.20)
Iteración 4: BR0079 CAJ1943342 -> CAJ1927270 (Δ=-29,858.10 | costo total ≈ 188,619,782.10)
Iteración 5: BR0347 CAJ1945596 -> CAJ1927270 (Δ=-29,401.62 | costo total ≈ 188,590,380.48)
Iteración 6: BR0147 CAJ1431292 -> CAJ1399148 (Δ=-22,808.58 | costo total ≈ 188,567,571.90)
Iteración 7: BR0160 CAJ0708749 -> CAJ0443892 (Δ=-17,677.98 | costo total ≈ 188,549,893.92)
Iteración 8: BR0240 CAJ2229012 -> CAJ2132580 (Δ=-12,971.70 | costo total ≈ 188,536,922.22)
Iteración 9: BR0361 CAJ0708749 -> CAJ0443892 (Δ=-12,189.48 | costo total ≈ 188,524,732.74)


KeyboardInterrupt: 

#### **Comparación: Greedy vs. Búsqueda local**

In [9]:
solucion_inicial.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30166293.939999998
Costo flete: 179068800
Costo total: 209235093.94
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 3mm
Criterio elegido: Greedy 5 (maximizar utilización de pallet) - punto de partida
Número de tipos de cajas distintos: 78
Costo packaging: 27296782.739999976
Costo flete: 161227950
Costo total: 188524732.73999998
Utilización de pallet promedio: 0.9579706456578424
Utilización de caja promedio: 0.9471952523608305
Ahorro costo total: 9.89813%


Exportamos la solución mejorada al mismo formato usado por las soluciones greedy:

In [10]:
solucion_inicial.exportar_submmit(nombre_csv="7-busqueda_local_3mm")

# Variante: búsqueda local restringida a las cajas ya presentes en la solución

La búsqueda local anterior evalúa, para cada producto, **todas** sus cajas factibles (que según `5.factibilidad.ipynb` pueden ser miles). Eso hace que una sola pasada sea muy costosa: en la corrida de arriba hubo que interrumpirla manualmente después de 9 movimientos.

Esta variante reduce drásticamente el vecindario: para cada producto solo se consideran como candidatas las **cajas que ya están siendo utilizadas por la solución** (`solucion.tipos_cajas_utilizados`), quedándose únicamente con aquellas que además aparecen en el csv de factibilidad para ese producto. Como el csv de factibilidad ya trae la lista de cajas asignables por producto (chequeos de dimensión, headspace, resistencia y utilización de pallet ≥ 0.6), el filtro se reduce a una consulta de pertenencia.

Pasamos así de ~427 productos × miles de candidatas a ~427 × la cantidad de tipos de caja de la solución (76 en el punto de partida), es decir unas ~32.000 evaluaciones de delta por pasada, lo que permite correr muchas más iteraciones hasta el óptimo local.

Además tiene un efecto colateral deseable: como nunca se incorpora un tipo de caja nuevo, la cantidad de tipos de caja de la solución **no puede aumentar**, y baja cada vez que un movimiento deja una caja sin productos. Esto empuja el volumen hacia menos tipos de caja, lo que ayuda a los descuentos por volumen del packaging.

La contracara es que el espacio de búsqueda es más chico: si la mejora requiere estrenar un tipo de caja que hoy no está en la solución, esta variante no la va a encontrar. Por eso conviene pensarla como complemento (barata y rápida) de la versión completa, no como reemplazo.

In [5]:
def construir_factibles_por_producto(factibilidad):
    """
    Devuelve {codigo_producto: set(caja_tipo_id)}, para poder preguntar si una caja
    es factible para un producto en O(1) en vez de recorrer toda la lista.

    Acepta:
      - el diccionario `cajas_asignables_por_producto` que arma `guardar_cajas_y_productos`
      - un DataFrame en formato largo: una fila por (codigo_producto, caja_tipo_id)
      - un DataFrame en formato ancho: una fila por producto y los ids en la columna
        `cajas_asignables_id` separados por '; '
    """
    if isinstance(factibilidad, dict):
        return {codigo: set(cajas_ids) for codigo, cajas_ids in factibilidad.items()}

    if 'caja_tipo_id' in factibilidad.columns:
        return {
            codigo: set(group['caja_tipo_id'])
            for codigo, group in factibilidad.groupby('codigo_producto')
        }

    return {
        row['codigo_producto']: {caja_id.strip() for caja_id in row['cajas_asignables_id'].split(';')}
        for _, row in factibilidad.iterrows()
    }


def busqueda_local_cajas_en_solucion(solucion, cajas_asignables_por_producto, max_iteraciones=500,
                                     verbose=True):
    """
    Búsqueda local por 'mejor mejora' (steepest descent), igual que `busqueda_local`,
    pero con el vecindario restringido: para cada producto solo se evalúan como
    candidatas las cajas que ya están en la solución y que además son factibles para
    ese producto según el csv de factibilidad.

    Modifica `solucion` in-place y devuelve un DataFrame con el historial de
    movimientos aplicados. Si se interrumpe manualmente (KeyboardInterrupt), los
    movimientos ya aplicados quedan y se devuelve el historial parcial.
    """
    factibles_por_producto = construir_factibles_por_producto(cajas_asignables_por_producto)

    historial = []
    costo_actual = solucion.costo_total()
    UMBRAL_MEJORA = 1e-6  # tolerancia para evitar ciclos por ruido de punto flotante
    iteracion = 0

    try:
        for iteracion in range(1, max_iteraciones + 1):
            mejor_delta = -UMBRAL_MEJORA
            mejor_movimiento = None  # (asignacion, caja_nueva)

            # Snapshot de las cajas de la solución al inicio de la pasada: son las
            # únicas candidatas. Se recalcula en cada pasada porque una caja puede
            # quedar vacía y salir de la solución.
            cajas_en_solucion = list(solucion.tipos_cajas_utilizados)

            for asignacion in solucion.asignaciones:
                producto = asignacion.producto
                caja_actual = asignacion.caja
                factibles = factibles_por_producto.get(producto.codigo_producto, set())

                for caja_candidata in cajas_en_solucion:
                    if caja_candidata is caja_actual:
                        continue

                    # Único chequeo necesario: que la caja aparezca en el csv de
                    # factibilidad para este producto
                    if caja_candidata.caja_id not in factibles:
                        continue

                    delta = calcular_delta_costo(producto, caja_actual, caja_candidata)

                    if delta < mejor_delta:
                        mejor_delta = delta
                        mejor_movimiento = (asignacion, caja_candidata)

            if mejor_movimiento is None:
                if verbose:
                    print(f"Iteración {iteracion}: no se encontraron mejoras. Óptimo local alcanzado.")
                break

            # Aplicamos el mejor movimiento encontrado en esta pasada
            asignacion, caja_nueva = mejor_movimiento
            producto = asignacion.producto
            caja_vieja = asignacion.caja

            caja_vieja.revocar_producto(producto)
            caja_nueva.asignar_producto(producto)
            asignacion.caja = caja_nueva

            # La caja nueva ya estaba en la solución, así que solo puede bajar la
            # cantidad de tipos de caja (cuando la vieja queda sin productos)
            if len(caja_vieja.productos_asignados) == 0 and caja_vieja in solucion.tipos_cajas_utilizados:
                solucion.tipos_cajas_utilizados.remove(caja_vieja)
                solucion.cantidad_tipos_cajas -= 1

            costo_actual += mejor_delta

            historial.append({
                "iteracion": iteracion,
                "codigo_producto": producto.codigo_producto,
                "caja_anterior": caja_vieja.caja_id,
                "caja_nueva": caja_nueva.caja_id,
                "delta_costo": mejor_delta,
                "costo_total_estimado": costo_actual,
                "tipos_cajas": solucion.cantidad_tipos_cajas
            })

            if verbose:
                print(f"Iteración {iteracion}: {producto.codigo_producto} "
                      f"{caja_vieja.caja_id} -> {caja_nueva.caja_id} "
                      f"(Δ={mejor_delta:,.2f} | costo total ≈ {costo_actual:,.2f} | "
                      f"tipos de caja: {solucion.cantidad_tipos_cajas})")

    except KeyboardInterrupt:
        if verbose:
            print(f"\nInterrumpido manualmente en la iteración {iteracion}. "
                  f"Se conservan los {len(historial)} movimientos ya aplicados.")

    return pd.DataFrame(historial)

#### **Ejecutamos la variante restringida**

Reconstruimos la solución del greedy desde cero (cajas y productos nuevos incluidos), porque los objetos `Caja` guardan estado de asignación y `solucion_inicial` ya fue modificada por la corrida anterior. Así las dos variantes arrancan del mismo punto y son comparables.

In [6]:
cajas_r, productos_r, cajas_asignables_por_producto_r = guardar_cajas_y_productos(grosor=grosor)

indice_cajas_por_dim_r = {
    (c.dim_interior_ancho, c.dim_interior_largo, c.dim_interior_alto): c
    for c in cajas_r.values()
}

solucion_restringida = Solucion(grosor, "Búsqueda local restringida a las cajas de la solución")

for _, row in solucion.iterrows():
    producto = productos_r[row["codigo_producto"]]

    dim_ancho = row["caja_exterior_ancho"] - 2 * row["caja_grosor_mm"]
    dim_largo = row["caja_exterior_largo"] - 2 * row["caja_grosor_mm"]
    dim_alto = row["caja_exterior_alto"] - 2 * row["caja_grosor_mm"]

    caja = indice_cajas_por_dim_r[(dim_ancho, dim_largo, dim_alto)]

    solucion_restringida.agregar_asignacion(Asignacion(producto, caja))

solucion_restringida.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30166293.939999998
Costo flete: 179068800
Costo total: 209235093.94
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 3mm
Criterio elegido: Búsqueda local restringida a las cajas de la solución
Número de tipos de cajas distintos: 76
Costo packaging: 27406334.33999998
Costo flete: 161380650
Costo total: 188786984.33999997
Utilización de pallet promedio: 0.9593515126880586
Utilización de caja promedio: 0.9445527162171283
Ahorro costo total: 9.77279%


In [7]:
historial_restringida = busqueda_local_cajas_en_solucion(
    solucion_restringida,
    cajas_asignables_por_producto_r,
    max_iteraciones=500,
    verbose=True
)

historial_restringida

Iteración 1: BR0239 CAJ2229012 -> CAJ2132580 (Δ=-50,991.00 | costo total ≈ 188,735,993.34 | tipos de caja: 76)
Iteración 2: BR0159 CAJ0708749 -> CAJ0443990 (Δ=-36,403.14 | costo total ≈ 188,699,590.20 | tipos de caja: 76)
Iteración 3: BR0079 CAJ1943342 -> CAJ1927270 (Δ=-29,858.10 | costo total ≈ 188,669,732.10 | tipos de caja: 76)
Iteración 4: BR0347 CAJ1945596 -> CAJ1927270 (Δ=-29,401.62 | costo total ≈ 188,640,330.48 | tipos de caja: 76)
Iteración 5: BR0147 CAJ1431292 -> CAJ1399148 (Δ=-22,808.58 | costo total ≈ 188,617,521.90 | tipos de caja: 76)
Iteración 6: BR0160 CAJ0708749 -> CAJ0443990 (Δ=-17,677.98 | costo total ≈ 188,599,843.92 | tipos de caja: 76)
Iteración 7: BR0361 CAJ0708749 -> CAJ0443990 (Δ=-48,139.50 | costo total ≈ 188,551,704.42 | tipos de caja: 76)
Iteración 8: BR0240 CAJ2229012 -> CAJ2132580 (Δ=-12,971.70 | costo total ≈ 188,538,732.72 | tipos de caja: 76)
Iteración 9: BR0363 CAJ2572404 -> CAJ2347396 (Δ=-11,593.02 | costo total ≈ 188,527,139.70 | tipos de caja: 76)
I

,iteracion,codigo_producto,caja_anterior,caja_nueva,delta_costo,costo_total_estimado,tipos_cajas
0,1,BR0239,CAJ2229012,CAJ2132580,-50991.00,1.887360e+08,76
1,2,BR0159,CAJ0708749,CAJ0443990,-36403.14,1.886996e+08,76
2,3,BR0079,CAJ1943342,CAJ1927270,-29858.10,1.886697e+08,76
3,4,BR0347,CAJ1945596,CAJ1927270,-29401.62,1.886403e+08,76
4,5,BR0147,CAJ1431292,CAJ1399148,-22808.58,1.886175e+08,76
...,...,...,...,...,...,...,...
71,72,BR0086,CAJ1811140,CAJ1730780,-13.08,1.882792e+08,60
72,73,BR0246,CAJ1939716,CAJ1891500,-12.24,1.882792e+08,60
73,74,BR0089,CAJ1939716,CAJ1891500,-9.24,1.882792e+08,59
74,75,BR0048,CAJ1923644,CAJ1891500,-4.74,1.882792e+08,59


In [8]:
solucion_restringida.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30166293.939999998
Costo flete: 179068800
Costo total: 209235093.94
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 3mm
Criterio elegido: Búsqueda local restringida a las cajas de la solución
Número de tipos de cajas distintos: 58
Costo packaging: 27034413.41999998
Costo flete: 161244750
Costo total: 188279163.42
Utilización de pallet promedio: 0.951583669746654
Utilización de caja promedio: 0.9536835848633282
Ahorro costo total: 10.01550%


In [9]:
solucion_restringida.exportar_submmit(nombre_csv="11-relocateRestringido_3mm")

# Swap sobre la solución restringida

`calcular_delta_costo` no sirve para evaluar un swap: calcula el efecto de mover un único producto y, por lo tanto, deja el volumen de la otra caja sin el producto que debería salir. Para el intercambio usamos `calcular_delta_costo_swap`, equivalente a la versión de `7b.swap&relocate.ipynb`.

La función calcula de forma no destructiva el packaging total de las dos cajas antes y después del intercambio. Así incorpora el costo unitario y el descuento por volumen de cada planta. También recalcula los pallets de ambos productos en sus nuevas cajas, de modo que el delta incluye el flete. La búsqueda solo acepta un par cuando la factibilidad es cruzada: cada producto debe poder usar la caja que actualmente usa el otro.

In [9]:
from Clases.caja import calcular_descuento_por_volumen

PLANTAS_SWAP = ["buenos_aires", "curitiba", "santiago", "monterrey", "bakersfield"]


def costo_packaging_caja_hipotetico(caja, deltas_unidades):
    """Costo de packaging de una caja luego de aplicar deltas por planta.

    No modifica la caja: los descuentos se recalculan solamente para el
    escenario hipotético.
    """
    return sum(
        (unidades := getattr(caja, f"unidades_{planta}_req") + deltas_unidades.get(planta, 0))
        * caja.costo_unitario
        * (1 + calcular_descuento_por_volumen(unidades))
        for planta in PLANTAS_SWAP
    )


def calcular_delta_costo_swap(producto_a, caja_a, producto_b, caja_b):
    """Delta de costo total al intercambiar las cajas de A y B.

    Precondición: A está asignado a caja_a, B a caja_b y las cajas son
    distintas. La función no muta la solución.
    """
    if caja_a is caja_b:
        return 0.0

    demanda_a = {planta: getattr(producto_a, f"demanda_{planta}") for planta in PLANTAS_SWAP}
    demanda_b = {planta: getattr(producto_b, f"demanda_{planta}") for planta in PLANTAS_SWAP}

    costo_packaging_antes = (
        costo_packaging_caja_hipotetico(caja_a, {})
        + costo_packaging_caja_hipotetico(caja_b, {})
    )

    # En caja_a sale A y entra B; en caja_b ocurre el cambio inverso.
    delta_caja_a = {planta: demanda_b[planta] - demanda_a[planta] for planta in PLANTAS_SWAP}
    delta_caja_b = {planta: demanda_a[planta] - demanda_b[planta] for planta in PLANTAS_SWAP}
    costo_packaging_despues = (
        costo_packaging_caja_hipotetico(caja_a, delta_caja_a)
        + costo_packaging_caja_hipotetico(caja_b, delta_caja_b)
    )

    delta_packaging = costo_packaging_despues - costo_packaging_antes

    # Cada producto puede requerir una cantidad distinta de pallets en su nueva caja.
    delta_flete = 150 * (
        Asignacion(producto_a, caja_b).cant_pallets_requeridas()
        - Asignacion(producto_a, caja_a).cant_pallets_requeridas()
        + Asignacion(producto_b, caja_a).cant_pallets_requeridas()
        - Asignacion(producto_b, caja_b).cant_pallets_requeridas()
    )

    return delta_packaging + delta_flete

In [10]:
def busqueda_local_swap_restringida(solucion, cajas_asignables_por_producto,
                                    max_iteraciones=100, verbose=True):
    """Mejor mejora mediante swaps entre pares de `solucion`.

    Para cada par se toma la caja actualmente usada por el otro producto y
    se chequea la factibilidad cruzada con `cajas_asignables_por_producto`.
    Solo se aplica el mejor swap de costo negativo por iteración.
    """
    factibles_por_producto = construir_factibles_por_producto(cajas_asignables_por_producto)
    historial = []
    costo_actual = solucion.costo_total()
    umbral_mejora = 1e-6

    for iteracion in range(1, max_iteraciones + 1):
        mejor_delta = -umbral_mejora
        mejor_swap = None
        asignaciones = solucion.asignaciones

        for i, asignacion_a in enumerate(asignaciones):
            producto_a, caja_a = asignacion_a.producto, asignacion_a.caja
            factibles_a = factibles_por_producto.get(producto_a.codigo_producto, set())

            for asignacion_b in asignaciones[i + 1:]:
                producto_b, caja_b = asignacion_b.producto, asignacion_b.caja

                if caja_a is caja_b:
                    continue  # El intercambio no cambia la solución.

                # Factibilidad cruzada: A usa la caja de B y B la caja de A.
                if (caja_b.caja_id not in factibles_a
                        or caja_a.caja_id not in factibles_por_producto.get(producto_b.codigo_producto, set())):
                    continue

                delta = calcular_delta_costo_swap(producto_a, caja_a, producto_b, caja_b)
                if delta < mejor_delta:
                    mejor_delta = delta
                    mejor_swap = (asignacion_a, asignacion_b)

        if mejor_swap is None:
            if verbose:
                print(f"Iteración {iteracion}: no se encontraron swaps que mejoren el costo. Óptimo local alcanzado.")
            break

        asignacion_a, asignacion_b = mejor_swap
        producto_a, caja_a = asignacion_a.producto, asignacion_a.caja
        producto_b, caja_b = asignacion_b.producto, asignacion_b.caja

        # Mutación real: se aplica solamente después de elegir el mejor par.
        caja_a.revocar_producto(producto_a)
        caja_b.revocar_producto(producto_b)
        caja_b.asignar_producto(producto_a)
        caja_a.asignar_producto(producto_b)
        asignacion_a.caja, asignacion_b.caja = caja_b, caja_a

        costo_actual += mejor_delta
        historial.append({
            "iteracion": iteracion,
            "codigo_producto_a": producto_a.codigo_producto,
            "caja_anterior_a": caja_a.caja_id,
            "caja_nueva_a": caja_b.caja_id,
            "codigo_producto_b": producto_b.codigo_producto,
            "caja_anterior_b": caja_b.caja_id,
            "caja_nueva_b": caja_a.caja_id,
            "delta_costo": mejor_delta,
            "costo_total_estimado": costo_actual,
        })

        if verbose:
            print(
                f"Iteración {iteracion}: swap {producto_a.codigo_producto} ({caja_a.caja_id}) "
                f"<-> {producto_b.codigo_producto} ({caja_b.caja_id}) "
                f"(Δ={mejor_delta:,.2f} | costo total ≈ {costo_actual:,.2f})"
            )

    return pd.DataFrame(historial)

#### Ejecutar swaps sobre `solucion_restringida`

Esta celda continúa desde el resultado de `busqueda_local_cajas_en_solucion`; no reconstruye ni modifica las celdas previas. El máximo de 100 iteraciones es un tope de seguridad: la ejecución termina antes si no encuentra una mejora.

In [11]:
historial_swap_restringida = busqueda_local_swap_restringida(
    solucion_restringida,
    cajas_asignables_por_producto_r,
    max_iteraciones=100,
    verbose=True
)

solucion_restringida.resumen_general()
historial_swap_restringida

Iteración 1: no se encontraron swaps que mejoren el costo. Óptimo local alcanzado.
Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30166293.939999998
Costo flete: 179068800
Costo total: 209235093.94
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 3mm
Criterio elegido: Búsqueda local restringida a las cajas de la solución
Número de tipos de cajas distintos: 58
Costo packaging: 27034413.41999998
Costo flete: 161244750
Costo total: 188279163.42
Utilización de pallet promedio: 0.951583669746654
Utilización de caja promedio: 0.9536835848633282
Ahorro costo total: 10.01550%


""


# Perturbaciones: ruin & recreate

El relocate restringido y el swap solo recorren cajas que ya pertenecen a la solución. Esta búsqueda iterada sale de ese vecindario: en cada intento elige varios productos de una caja usada, los reasigna a cajas factibles que **no** estaban en la solución, y luego repara con relocate y swap.

Cada intento se ejecuta sobre una copia liviana de la mejor solución vigente. Las cajas se clonan solo cuando se necesitan, por lo que no se reconstruye el catálogo completo de cajas. Si el intento no mejora, se descarta sin alterar `solucion_restringida`; si mejora, pasa a ser la nueva base del siguiente intento.

In [ ]:
import random


def calcular_delta_costo_relocate_puro(producto, caja_actual, caja_nueva):
    """Delta de un relocate sin mutar cajas durante la evaluación."""
    if caja_actual is caja_nueva:
        return 0.0

    demanda = {planta: getattr(producto, f"demanda_{planta}") for planta in PLANTAS_SWAP}
    costo_antes = (costo_packaging_caja_hipotetico(caja_actual, {})
                   + costo_packaging_caja_hipotetico(caja_nueva, {}))
    costo_despues = (costo_packaging_caja_hipotetico(
                        caja_actual, {planta: -unidades for planta, unidades in demanda.items()})
                      + costo_packaging_caja_hipotetico(caja_nueva, demanda))

    delta_packaging = costo_despues - costo_antes
    delta_flete = 150 * (
        Asignacion(producto, caja_nueva).cant_pallets_requeridas()
        - Asignacion(producto, caja_actual).cant_pallets_requeridas()
    )
    return delta_packaging + delta_flete


def mover_producto(solucion, asignacion, caja_nueva):
    """Aplica un relocate y mantiene consistente el tracking de cajas usadas."""
    caja_vieja = asignacion.caja
    producto = asignacion.producto

    caja_vieja.revocar_producto(producto)
    caja_nueva.asignar_producto(producto)
    asignacion.caja = caja_nueva

    if caja_nueva not in solucion.tipos_cajas_utilizados:
        solucion.tipos_cajas_utilizados.append(caja_nueva)
        solucion.cantidad_tipos_cajas += 1
    if not caja_vieja.productos_asignados and caja_vieja in solucion.tipos_cajas_utilizados:
        solucion.tipos_cajas_utilizados.remove(caja_vieja)
        solucion.cantidad_tipos_cajas -= 1


def busqueda_local_restringida_pura(solucion, cajas_asignables_por_producto,
                                     max_iteraciones=30):
    """Relocate por mejor mejora dentro de las cajas presentes en `solucion`."""
    factibles = construir_factibles_por_producto(cajas_asignables_por_producto)
    umbral_mejora = 1e-6
    movimientos = 0

    for _ in range(max_iteraciones):
        mejor_delta = -umbral_mejora
        mejor_movimiento = None
        cajas_en_solucion = list(solucion.tipos_cajas_utilizados)

        for asignacion in solucion.asignaciones:
            producto = asignacion.producto
            caja_actual = asignacion.caja
            for caja_candidata in cajas_en_solucion:
                if (caja_candidata is caja_actual
                        or caja_candidata.caja_id not in factibles.get(producto.codigo_producto, set())):
                    continue

                delta = calcular_delta_costo_relocate_puro(producto, caja_actual, caja_candidata)
                if delta < mejor_delta:
                    mejor_delta = delta
                    mejor_movimiento = (asignacion, caja_candidata)

        if mejor_movimiento is None:
            break

        mover_producto(solucion, *mejor_movimiento)
        movimientos += 1

    return movimientos


def clonar_caja_vacia(caja):
    """Clona solo la geometría y el grosor; no copia asignaciones ni descuentos."""
    clon = Caja(caja.caja_id, caja.dim_interior_ancho, caja.dim_interior_largo, caja.dim_interior_alto)
    clon.elegir_grosor(caja.grosor_mm)
    return clon


def clonar_solucion_liviana(solucion_base, titulo):
    """Copia una solución sin duplicar todo el catálogo de cajas."""
    cajas_clonadas = {}
    solucion_clon = Solucion(solucion_base.grosor_elegido, titulo)

    for asignacion in solucion_base.asignaciones:
        caja_origen = asignacion.caja
        if caja_origen.caja_id not in cajas_clonadas:
            cajas_clonadas[caja_origen.caja_id] = clonar_caja_vacia(caja_origen)
        solucion_clon.agregar_asignacion(
            Asignacion(asignacion.producto, cajas_clonadas[caja_origen.caja_id])
        )

    return solucion_clon, cajas_clonadas


def elegir_caja_nueva(producto, ids_usados, factibles_por_producto, cajas_catalogo,
                      rng, max_candidatos=30):
    """Elige una caja factible no usada mediante una muestra dirigida por flete."""
    candidatas = [
        caja_id for caja_id in factibles_por_producto.get(producto.codigo_producto, set())
        if caja_id not in ids_usados
    ]
    if not candidatas:
        return None

    muestra = rng.sample(candidatas, min(max_candidatos, len(candidatas)))
    # La muestra aleatoria aporta diversidad; dentro de ella priorizamos menos pallets.
    return min(
        muestra,
        key=lambda caja_id: Asignacion(producto, cajas_catalogo[caja_id]).cant_pallets_requeridas()
    )


def seleccionar_ruina(solucion, tamanio_ruina, rng):
    """Selecciona primero una caja y luego completa con productos aleatorios."""
    por_caja = {}
    for asignacion in solucion.asignaciones:
        por_caja.setdefault(asignacion.caja.caja_id, []).append(asignacion)

    seleccionadas = list(por_caja[rng.choice(list(por_caja))])
    rng.shuffle(seleccionadas)
    seleccionadas = seleccionadas[:tamanio_ruina]

    if len(seleccionadas) < tamanio_ruina:
        restantes = [a for a in solucion.asignaciones if a not in seleccionadas]
        seleccionadas.extend(rng.sample(restantes, min(tamanio_ruina - len(seleccionadas), len(restantes))))

    return seleccionadas


def busqueda_ruin_recreate(solucion_inicial, cajas_catalogo, cajas_asignables_por_producto,
                           n_intentos=20, tamanio_ruina=5,
                           max_candidatos_nuevos=30, max_iter_reparacion=30,
                           max_iter_swap=30, semilla=42, verbose=True):
    """Búsqueda iterada que introduce cajas nuevas y conserva la mejor solución."""
    rng = random.Random(semilla)
    factibles = construir_factibles_por_producto(cajas_asignables_por_producto)
    mejor_solucion, _ = clonar_solucion_liviana(solucion_inicial, "Mejor solución con perturbaciones")
    mejor_costo = mejor_solucion.costo_total()
    historial = []

    for intento in range(1, n_intentos + 1):
        candidata, cajas_clonadas = clonar_solucion_liviana(
            mejor_solucion, f"Perturbación {intento}"
        )
        costo_base = candidata.costo_total()
        ids_usados = {caja.caja_id for caja in candidata.tipos_cajas_utilizados}
        productos_perturbados = []
        cajas_nuevas = []

        for asignacion in seleccionar_ruina(candidata, tamanio_ruina, rng):
            producto = asignacion.producto
            caja_id_nueva = elegir_caja_nueva(
                producto, ids_usados, factibles, cajas_catalogo, rng, max_candidatos_nuevos
            )
            if caja_id_nueva is None:
                continue

            if caja_id_nueva not in cajas_clonadas:
                cajas_clonadas[caja_id_nueva] = clonar_caja_vacia(cajas_catalogo[caja_id_nueva])
            mover_producto(candidata, asignacion, cajas_clonadas[caja_id_nueva])
            productos_perturbados.append(producto.codigo_producto)
            cajas_nuevas.append(caja_id_nueva)

        if not productos_perturbados:
            historial.append({"intento": intento, "aceptada": False, "motivo": "sin_caja_nueva_factible"})
            continue

        costo_post_ruina = candidata.costo_total()
        relocates = busqueda_local_restringida_pura(
            candidata, cajas_asignables_por_producto, max_iteraciones=max_iter_reparacion
        )
        swaps = busqueda_local_swap_restringida(
            candidata, cajas_asignables_por_producto, max_iteraciones=max_iter_swap, verbose=False
        )
        costo_final = candidata.costo_total()
        aceptada = costo_final < mejor_costo - 1e-6

        historial.append({
            "intento": intento,
            "aceptada": aceptada,
            "costo_base": costo_base,
            "costo_post_ruina": costo_post_ruina,
            "costo_final": costo_final,
            "delta_vs_base": costo_final - costo_base,
            "productos_perturbados": ", ".join(productos_perturbados),
            "cajas_nuevas_probadas": ", ".join(map(str, sorted(set(cajas_nuevas)))),
            "relocates_reparacion": relocates,
            "swaps_reparacion": len(swaps),
            "tipos_cajas_final": candidata.cantidad_tipos_cajas,
        })

        if aceptada:
            mejor_solucion = candidata
            mejor_costo = costo_final
            if verbose:
                print(f"Intento {intento}: mejora aceptada (Δ={costo_final - costo_base:,.2f}; costo={mejor_costo:,.2f})")
        elif verbose:
            print(f"Intento {intento}: sin mejora (Δ={costo_final - costo_base:,.2f})")

    return mejor_solucion, pd.DataFrame(historial)

#### Ejecutar la búsqueda con perturbaciones

Este primer barrido usa 20 intentos de cinco productos. Para explorar más lejos, conviene repetir con otra semilla o aumentar `tamanio_ruina` a 8 y `n_intentos` a 50. La variable `solucion_restringida` se reemplaza solo al terminar, con la mejor solución hallada.

In [ ]:
solucion_restringida, historial_perturbaciones = busqueda_ruin_recreate(
    solucion_restringida,
    cajas_r,
    cajas_asignables_por_producto_r,
    n_intentos=20,
    tamanio_ruina=5,
    max_candidatos_nuevos=30,
    max_iter_reparacion=30,
    max_iter_swap=30,
    semilla=42,
    verbose=True
)

solucion_restringida.resumen_general()
historial_perturbaciones